In [ ]:
import cv2
import numpy as np
import json

# ================================
# 1. LOAD CAMERA CALIBRATION
# ================================
calib = np.load(r"E:\Sanjay\Dodeca\realsense_color_intrinsics_1280x720.npz")
K = calib["camera_matrix"]
D = calib["dist_coeffs"]

# ==========================================
# 2. LOAD CALIBRATED MARKER GEOMETRY (DC)
# ==========================================

with open(r"E:\Sanjay\Dodeca\optimized_marker_object_poses.json", "r") as f:
    marker_geometry = json.load(f)

# Convert lists → numpy arrays in place
for key in marker_geometry:
    marker_geometry[key]["R"] = np.array(marker_geometry[key]["R"], dtype=np.float32)
    marker_geometry[key]["t"] = np.array(marker_geometry[key]["t"], dtype=np.float32)

print("All markers converted to numpy arrays")
print("Available markers:", list(marker_geometry.keys()))

In [ ]:
# ==============================
# ARUCO INITIALIZATION
# ==============================
# Choose dictionary (must match printed markers)
aruco_dict = cv2.aruco.getPredefinedDictionary(
    cv2.aruco.DICT_6X6_250   # Change if you used different dictionary
)

# Detection parameters
aruco_params = cv2.aruco.DetectorParameters()

# Create detector
aruco_detector = cv2.aruco.ArucoDetector(aruco_dict, aruco_params)

# ==========================================
# 4. MOTION MODEL INITIALIZATION
# ==========================================
previous_pose = None
previous_velocity = np.zeros(6)
previous_acceleration = np.zeros(6)


REPROJECTION_THRESHOLD = 3.0
# ==========================================
# 5. ARUCO DETECTION FUNCTION
# ==========================================
def detect_aruco(frame):

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    corners, ids, rejected = aruco_detector.detectMarkers(gray)

    if ids is None:
        return [], {}

    detected_ids = ids.flatten().tolist()
    detected_corners_dict = {}

    for i, marker_id in enumerate(detected_ids):
        detected_corners_dict[marker_id] = corners[i][0]

    return detected_ids, detected_corners_dict
# ==========================================
# 6. LOAD VIDEO
# ==========================================
video_path = r"E:\Sanjay\Dodeca\realsense_recording_1280x720_30fps.mp4"  # change path
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error opening video")
    exit()

print("Video loaded")

In [ ]:
def build_correspondences(detected_ids,
                          detected_corners,
                          marker_geometry,
                          marker_size=20.0):
    """
    Build 3D–2D correspondences using DC-calibrated marker poses.

    marker_size in millimeters (must match DC calibration).
    """

    object_points = []
    image_points = []

    s = marker_size / 2.0

    # EXACT SAME ORDER USED IN DC CALIBRATION
    local_corners = np.array([
        [-s, -s, 0],   # bottom-left
        [ s, -s, 0],   # bottom-right
        [ s,  s, 0],   # top-right
        [-s,  s, 0]    # top-left
    ], dtype=np.float32)

    for marker_id in detected_ids:

        marker_id_str = str(marker_id)

        if marker_id_str not in marker_geometry:
            continue

        R = marker_geometry[marker_id_str]["R"]   # 3x3
        t = marker_geometry[marker_id_str]["t"]   # 3,

        img_corners = detected_corners[marker_id]

        # Transform local marker corners → object frame
        for i in range(4):

            corner_obj = R @ local_corners[i] + t

            object_points.append(corner_obj)
            image_points.append(img_corners[i])

    object_points = np.array(object_points, dtype=np.float32)
    image_points = np.array(image_points, dtype=np.float32)

    if len(object_points) < 8:
        return None, None, False

    return object_points, image_points, True

In [ ]:
def project_dodecahedron_bbox(rvec, tvec, marker_geometry, K, D):
    """
    Project all marker corners to image and compute bounding box.
    """

    all_points_3d = []

    marker_size = 20.0
    s = marker_size / 2.0

    local_corners = np.array([
        [-s, -s, 0],
        [ s, -s, 0],
        [ s,  s, 0],
        [-s,  s, 0]
    ], dtype=np.float32)

    for key in marker_geometry:
        R = marker_geometry[key]["R"]
        t = marker_geometry[key]["t"]

        for i in range(4):
            pt = R @ local_corners[i] + t
            all_points_3d.append(pt)

    all_points_3d = np.array(all_points_3d, dtype=np.float32)

    projected, _ = cv2.projectPoints(all_points_3d, rvec, tvec, K, D)
    projected = projected.reshape(-1, 2)

    xmin = int(np.min(projected[:, 0]))
    xmax = int(np.max(projected[:, 0]))
    ymin = int(np.min(projected[:, 1]))
    ymax = int(np.max(projected[:, 1]))

    return xmin, ymin, xmax, ymax

## APE with constant accerleration step as per paper 

In [ ]:
while True:

    ret, frame = cap.read()
    if not ret:
        break

    h, w = frame.shape[:2]

    # =========================================================
    # STEP 1 — PREDICT POSE (Constant Acceleration Model)
    # =========================================================
    use_guess = False
    init_rvec = None
    init_tvec = None

    if previous_pose is not None:

        predicted_pose = previous_pose + previous_velocity + 0.5 * previous_acceleration

        init_rvec = predicted_pose[:3].reshape(3,1)
        init_tvec = predicted_pose[3:].reshape(3,1)

        use_guess = True

        # =====================================================
        # STEP 2 — SEARCH REGION CONSTRAINT
        # =====================================================
        xmin, ymin, xmax, ymax = project_dodecahedron_bbox(
            init_rvec, init_tvec, marker_geometry, K, D
        )

        # Expand region to 4× area
        width = xmax - xmin
        height = ymax - ymin

        xmin = max(0, xmin - width)
        xmax = min(w, xmax + width)
        ymin = max(0, ymin - height)
        ymax = min(h, ymax + height)

        roi = frame[ymin:ymax, xmin:xmax]

        detected_ids, detected_corners = detect_aruco(roi)

        # Convert ROI coordinates back to full image coordinates
        for marker_id in detected_corners:
            detected_corners[marker_id][:, 0] += xmin
            detected_corners[marker_id][:, 1] += ymin

    else:
        detected_ids, detected_corners = detect_aruco(frame)

    if len(detected_ids) == 0:
        cv2.imshow("APE", frame)
        if cv2.waitKey(1) == 27:
            break
        continue

    # =========================================================
    # BUILD CORRESPONDENCES
    # =========================================================
    object_points, image_points, ok = build_correspondences(
        detected_ids,
        detected_corners,
        marker_geometry
    )

    if not ok:
        continue

    # =========================================================
    # STEP 3 — SOLVE PnP WITH EXTRINSIC GUESS
    # =========================================================
    success, rvec, tvec = cv2.solvePnP(
        object_points,
        image_points,
        K,
        D,
        rvec=init_rvec,
        tvec=init_tvec,
        useExtrinsicGuess=use_guess,
        flags=cv2.SOLVEPNP_ITERATIVE
    )

    if not success:
        continue

    # =========================================================
    # STEP 4 — REPROJECTION ERROR
    # =========================================================
    projected_points, _ = cv2.projectPoints(
        object_points,
        rvec,
        tvec,
        K,
        D
    )

    projected_points = projected_points.reshape(-1, 2)

    errors = np.linalg.norm(image_points - projected_points, axis=1)
    mean_error = np.mean(errors)

    if mean_error > REPROJECTION_THRESHOLD:
        print("Pose rejected ❌  Error:", mean_error)
        continue

    # =========================================================
    # STEP 5 — UPDATE MOTION MODEL
    # =========================================================
    current_pose = np.concatenate((rvec.flatten(), tvec.flatten()))

    if previous_pose is not None:
        velocity = current_pose - previous_pose
        acceleration = velocity - previous_velocity
    else:
        velocity = np.zeros(6)
        acceleration = np.zeros(6)

    previous_pose = current_pose
    previous_velocity = velocity
    previous_acceleration = acceleration

    # =========================================================
    # DRAW OUTPUT
    # =========================================================
    cv2.drawFrameAxes(frame, K, D, rvec, tvec, 40)

    print("Pose accepted ✅  |  Error:", mean_error)

    cv2.imshow("APE", frame)

    if cv2.waitKey(1) == 27:
        break

cap.release()
cv2.destroyAllWindows()

## Aproximate pose estiamtion with kalman filter

In [ ]:
import cv2
import numpy as np
import json
import pyrealsense2 as rs

# ================================
# 1. LOAD CAMERA & GEOMETRY
# ================================
calib = np.load(r"E:\Sanjay\Dodeca\realsense_color_intrinsics_1280x720.npz")
K = calib["camera_matrix"]
D = calib["dist_coeffs"]

with open(r"E:\Sanjay\Dodeca\optimized_marker_object_poses.json", "r") as f:
    marker_geometry = json.load(f)

for key in marker_geometry:
    marker_geometry[key]["R"] = np.array(marker_geometry[key]["R"], dtype=np.float32)
    marker_geometry[key]["t"] = np.array(marker_geometry[key]["t"], dtype=np.float32)

# ==============================
# 2. SETTINGS & KALMAN INIT
# ==============================
REPROJECTION_THRESHOLD = 5.0  # Tightened slightly for better accuracy

kf = cv2.KalmanFilter(12, 6) # [x, y, z, rx, ry, rz, vx, vy, vz, vrx, vry, vrz]
kf.transitionMatrix = np.eye(12, dtype=np.float32)
for i in range(6):
    kf.transitionMatrix[i, i+6] = 1.0  # Constant velocity model

kf.measurementMatrix = np.zeros((6, 12), np.float32) # [x, y, z, rx, ry, rz]
for i in range(6):
    kf.measurementMatrix[i, i] = 1.0

kf.processNoiseCov = np.eye(12, dtype=np.float32) * 0.01 
kf.measurementNoiseCov = np.eye(6, dtype=np.float32) * 0.1
kf.errorCovPost = np.eye(12, dtype=np.float32)

# ==============================
# 3. HELPER FUNCTIONS
# ==============================
aruco_detector = cv2.aruco.ArucoDetector(
    cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_6X6_250),
    cv2.aruco.DetectorParameters()
)

def build_correspondences(detected_ids, detected_corners, marker_geometry, marker_size=20.0):
    obj_pts, img_pts = [], []
    s = marker_size / 2.0
    lc = np.array([[-s,-s,0],[s,-s,0],[s,s,0],[-s,s,0]], dtype=np.float32)
    for m_id in detected_ids:
        m_str = str(m_id)
        if m_str not in marker_geometry: continue
        R_m, t_m = marker_geometry[m_str]["R"], marker_geometry[m_str]["t"]
        for i in range(4):
            obj_pts.append(R_m @ lc[i] + t_m)
            img_pts.append(detected_corners[m_id][i])
    return np.array(obj_pts, np.float32), np.array(img_pts, np.float32), len(obj_pts) >= 4

def get_bbox(rvec, tvec, marker_geometry, K, D):
    all_pts = []
    s = 10.0
    lc = np.array([[-s,-s,0],[s,-s,0],[s,s,0],[-s,s,0]], dtype=np.float32)
    for k in marker_geometry:
        for i in range(4): all_pts.append(marker_geometry[k]["R"] @ lc[i] + marker_geometry[k]["t"])
    proj, _ = cv2.projectPoints(np.array(all_pts), rvec, tvec, K, D)
    proj = proj.reshape(-1, 2)
    return int(np.min(proj[:,0])), int(np.min(proj[:,1])), int(np.max(proj[:,0])), int(np.max(proj[:,1]))

# ==============================
# 4. MAIN LOOP
# ==============================
pipeline = rs.pipeline()
config = rs.config()
config.enable_stream(rs.stream.color, 1280, 720, rs.format.bgr8, 30)
pipeline.start(config)

initialized = False

while True:
    frames = pipeline.wait_for_frames()
    frame = np.asanyarray(frames.get_color_frame().get_data())
    h, w = frame.shape[:2]

    # --- PREDICT ---
    prediction = kf.predict()
    pred_rvec = prediction[:3]
    pred_tvec = prediction[3:6]
    #pred_tvec = prediction[:3]
    #pred_rvec = prediction[3:6]

    # --- ROI LOGIC ---
    if initialized:
        try:
            xmin, ymin, xmax, ymax = get_bbox(pred_rvec, pred_tvec, marker_geometry, K, D)
            bw, bh = xmax - xmin, ymax - ymin
            xmin, xmax = max(0, xmin - bw), min(w, xmax + bw)
            ymin, ymax = max(0, ymin - bh), min(h, ymax + bh)
            roi = frame[ymin:ymax, xmin:xmax]
        except: # Catch projection errors if rvec is wild
            roi, xmin, ymin = frame, 0, 0
    else:
        roi, xmin, ymin = frame, 0, 0

    # --- DETECT ---
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    corners, ids, _ = aruco_detector.detectMarkers(gray)

    if ids is not None:
        ids_list = ids.flatten().tolist()
        corners_dict = {ids_list[i]: corners[i][0] + [xmin, ymin] for i in range(len(ids_list))}
        obj_p, img_p, ok = build_correspondences(ids_list, corners_dict, marker_geometry)

        if ok:
            # --- SOLVE PNP ---
            succ, rvec, tvec = cv2.solvePnP(obj_p, img_p, K, D, 
                                            rvec=pred_rvec, tvec=pred_tvec, 
                                            useExtrinsicGuess=initialized, 
                                            flags=cv2.SOLVEPNP_ITERATIVE)
            if succ:
                # --- REPROJECTION ERROR CHECK ---
                proj_p, _ = cv2.projectPoints(obj_p, rvec, tvec, K, D)
                mean_error = np.mean(np.linalg.norm(img_p - proj_p.reshape(-1, 2), axis=1))

                if mean_error < REPROJECTION_THRESHOLD:
                    # --- CORRECT (ONLY ON GOOD DATA) ---
                    measurement = np.concatenate((rvec, tvec)).astype(np.float32)
                    if not initialized:
                        kf.statePost[:6] = measurement
                        initialized = True
                    
                    estimated = kf.correct(measurement)
                    draw_r, draw_t = estimated[:3], estimated[3:6]
                    cv2.drawFrameAxes(frame, K, D, draw_r, draw_t, 40)
                    print(f"Pose accepted ✅ Error: {mean_error:.2f}")
                else:
                    print(f"Pose rejected ❌ High Error: {mean_error:.2f}")
                    # We DON'T set initialized=False here so the filter can keep predicting
            else:
                initialized = False
    else:
        initialized = False

    cv2.imshow("Filtered APE Tracking", frame)
    if cv2.waitKey(1) == 27: break

pipeline.stop()
cv2.destroyAllWindows()

## Apporximate pose estimation with ICT with 2 rounds of verification 

In [3]:
import cv2
import numpy as np
import json
import pyrealsense2 as rs

# ================================
# 1. LOAD CAMERA & GEOMETRY
# ================================
calib = np.load(r"E:\Sanjay\Dodeca\realsense_color_intrinsics_1280x720.npz")
K = calib["camera_matrix"]
D = calib["dist_coeffs"]

with open(r"E:\Sanjay\Dodeca\Rigid\optimized_marker_object_poses_ITERATIVE.json", "r") as f:
    marker_geometry = json.load(f)

for key in marker_geometry:
    marker_geometry[key]["R"] = np.array(marker_geometry[key]["R"], dtype=np.float32)
    marker_geometry[key]["t"] = np.array(marker_geometry[key]["t"], dtype=np.float32)

# ==============================
# 2. SETTINGS & KALMAN INIT
# ==============================
REPROJECTION_THRESHOLD = 8.0  
LK_PARAMS = dict(winSize=(21, 21), maxLevel=3, 
                 criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))

kf = cv2.KalmanFilter(12, 6) # [x, y, z, rx, ry, rz, vx, vy, vz, vrx, vry, vrz]
kf.transitionMatrix = np.eye(12, dtype=np.float32)
for i in range(6):
    kf.transitionMatrix[i, i+6] = 1.0  # Constant velocity model

kf.measurementMatrix = np.zeros((6, 12), np.float32) # [x, y, z, rx, ry, rz]
for i in range(6):
    kf.measurementMatrix[i, i] = 1.0

kf.processNoiseCov = np.eye(12, dtype=np.float32) * 0.01 
kf.measurementNoiseCov = np.eye(6, dtype=np.float32) * 0.1
kf.errorCovPost = np.eye(12, dtype=np.float32)

# ==============================
# 3. CORE FUNCTIONS
# ==============================
aruco_detector = cv2.aruco.ArucoDetector(
    cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_6X6_250),
    cv2.aruco.DetectorParameters()
)

def build_correspondences(corners_dict, marker_geometry, marker_size=20.0):
    obj_pts, img_pts = [], []
    s = marker_size / 2.0
    lc = np.array([[-s,-s,0],[s,-s,0],[s,s,0],[-s,s,0]], dtype=np.float32)
    for m_id, img_corners in corners_dict.items():
        m_str = str(m_id)
        if m_str not in marker_geometry: continue
        R_m, t_m = marker_geometry[m_str]["R"], marker_geometry[m_str]["t"]
        for i in range(4):
            obj_pts.append(R_m @ lc[i] + t_m)
            img_pts.append(img_corners[i])
    return np.array(obj_pts, np.float32), np.array(img_pts, np.float32), len(obj_pts) >= 8

def run_ICT(prev_gray, curr_gray, prev_corners_dict):
    """Refined Two-round Outlier Rejection ICT."""
    tracked_corners = {}
    if not prev_corners_dict: return tracked_corners

    # --- ROUND 1: Marker-level (Centers) ---
    marker_ids = list(prev_corners_dict.keys())
    prev_centers = np.array([np.mean(prev_corners_dict[mid], axis=0) for mid in marker_ids], dtype=np.float32).reshape(-1,1,2)
    
    next_centers, status, _ = cv2.calcOpticalFlowPyrLK(prev_gray, curr_gray, prev_centers, None, **LK_PARAMS)
    
    if next_centers is None or not any(status): return tracked_corners

    # Calculate velocity for centers that were successfully tracked
    valid_idx = np.where(status.flatten() == 1)[0]
    if len(valid_idx) < 1: return tracked_corners
    
    vel_centers = (next_centers[valid_idx] - prev_centers[valid_idx]).reshape(-1, 2)
    vel_mags = np.linalg.norm(vel_centers, axis=1)
    mean_v, std_v = np.mean(vel_mags), np.std(vel_mags)
    
    # --- ROUND 2: Corner-level (Trusted Markers & Corner Velocity Filtering) ---
    for i in valid_idx:
        m_id = marker_ids[i]
        
        # Round 1 Filter: Is this marker moving like the others?
        if abs(vel_mags[np.where(valid_idx==i)[0][0]] - mean_v) > 3 * std_v and len(valid_idx) > 2:
            continue
        
        p_corners = prev_corners_dict[m_id].reshape(-1, 1, 2).astype(np.float32)
        n_corners, c_status, _ = cv2.calcOpticalFlowPyrLK(prev_gray, curr_gray, p_corners, None, **LK_PARAMS)
        
        if n_corners is not None and all(c_status):
            # NEW: Round 2 Corner-level Velocity Filtering
            c_vel = (n_corners - p_corners).reshape(-1, 2)
            c_vel_mag = np.linalg.norm(c_vel, axis=1)
            
            mean_c, std_c = np.mean(c_vel_mag), np.std(c_vel_mag)
            # If std is tiny (stationary), avoid division issues or over-filtering
            mask = np.abs(c_vel_mag - mean_c) <= (3 * std_c + 0.1) 
            
            if np.all(mask): # Keep marker only if all 4 corners are consistent
                tracked_corners[m_id] = n_corners.reshape(4, 2)
    return tracked_corners

def get_bbox(rvec, tvec, marker_geometry, K, D):
    all_pts = []
    for k in marker_geometry:
        s = 10.0 
        lc = np.array([[-s,-s,0],[s,-s,0],[s,s,0],[-s,s,0]], dtype=np.float32)
        for i in range(4): all_pts.append(marker_geometry[k]["R"] @ lc[i] + marker_geometry[k]["t"])
    proj, _ = cv2.projectPoints(np.array(all_pts), rvec, tvec, K, D)
    proj = proj.reshape(-1, 2)
    return int(np.min(proj[:,0])), int(np.min(proj[:,1])), int(np.max(proj[:,0])), int(np.max(proj[:,1]))

# ==============================
# 4. MAIN LOOP
# ==============================
pipeline = rs.pipeline()
config = rs.config()
config.enable_stream(rs.stream.color, 1280, 720, rs.format.bgr8, 30)
pipeline.start(config)

initialized = False
prev_gray = None
prev_corners = {}

print("--- Starting Dodecahedron Tracking ---")
print(f"Reprojection Threshold: {REPROJECTION_THRESHOLD}px")

try:
    while True:
        frames = pipeline.wait_for_frames()
        frame = np.asanyarray(frames.get_color_frame().get_data())
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        h, w = frame.shape[:2]

        prediction = kf.predict()
        pred_rvec, pred_tvec = prediction[:3], prediction[3:6]

        if initialized:
            try:
                xmin, ymin, xmax, ymax = get_bbox(pred_rvec, pred_tvec, marker_geometry, K, D)
                bw, bh = xmax - xmin, ymax - ymin
                xmin, xmax = max(0, xmin - bw), min(w, xmax + bw)
                ymin, ymax = max(0, ymin - bh), min(h, ymax + bh)
                roi_gray = gray[ymin:ymax, xmin:xmax]
            except: roi_gray, xmin, ymin = gray, 0, 0
        else: roi_gray, xmin, ymin = gray, 0, 0

        corners, ids, _ = aruco_detector.detectMarkers(roi_gray)
        current_corners_candidate = {}
        pose_found = False

        # --- 3. APE SECTION ---
        if ids is not None:
            ids_list = ids.flatten().tolist()
            current_corners_candidate = {ids_list[i]: corners[i][0] + [xmin, ymin] for i in range(len(ids_list))}
            obj_p, img_p, ok = build_correspondences(current_corners_candidate, marker_geometry)
            
            if ok:
                succ, rvec, tvec = cv2.solvePnP(obj_p, img_p, K, D, rvec=pred_rvec.copy(), tvec=pred_tvec.copy(), 
                                                useExtrinsicGuess=initialized, flags=cv2.SOLVEPNP_SQPNP)
                if succ:
                    proj_p, _ = cv2.projectPoints(obj_p, rvec, tvec, K, D)
                    err = np.mean(np.linalg.norm(img_p - proj_p.reshape(-1, 2), axis=1))
                    if err < REPROJECTION_THRESHOLD:
                        pose_found = True
                        mode_text = "APE (ArUco)"
                        print(f"[APE] SUCCESS: Error = {err:.3f} px | Markers: {len(ids)}")
                    else:
                        print(f"[APE] REJECTED: Error too high ({err:.3f} px)")

        # --- 4. ICT SECTION ---
        if not pose_found and initialized and prev_gray is not None:
            tracked_dict = run_ICT(prev_gray, gray, prev_corners)
            obj_p, img_p, ok = build_correspondences(tracked_dict, marker_geometry)
            
            if ok:
                succ, rvec, tvec = cv2.solvePnP(obj_p, img_p, K, D, rvec=pred_rvec.copy(), tvec=pred_tvec.copy(), 
                                                useExtrinsicGuess=True, flags=cv2.SOLVEPNP_SQPNP)
                if succ:
                    proj_p, _ = cv2.projectPoints(obj_p, rvec, tvec, K, D)
                    err = np.mean(np.linalg.norm(img_p - proj_p.reshape(-1, 2), axis=1))
                    if err < REPROJECTION_THRESHOLD:
                        current_corners_candidate = tracked_dict 
                        pose_found = True
                        mode_text = "ICT (Optical Flow)"
                        print(f"[ICT] SUCCESS: Error = {err:.3f} px | Markers: {len(tracked_dict)}")
                    else:
                        print(f"[ICT] REJECTED: Error too high ({err:.3f} px)")

        # --- 5. UPDATE STATE ---
        if pose_found:
            measurement = np.concatenate((rvec, tvec)).astype(np.float32)
            if not initialized:
                kf.statePost[:6] = measurement
                initialized = True
            
            estimated = kf.correct(measurement)
            draw_r, draw_t = estimated[:3], estimated[3:6]
            cv2.drawFrameAxes(frame, K, D, draw_r, draw_t, 40)
            prev_corners = current_corners_candidate
            cv2.putText(frame, f"Mode: {mode_text}", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            cv2.putText(frame, f"Err: {err:.2f}", (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
        else:
            if initialized:
                print("[LOG] Tracking Lost - Predicting via Kalman Only")

        prev_gray = gray.copy()
        #prev_corners = final_corners_dict

        cv2.imshow("Dodecahedron Tracking (APE + ICT)", frame)
        if cv2.waitKey(1) == 27: break

finally:
    pipeline.stop()
    cv2.destroyAllWindows()

--- Starting Dodecahedron Tracking ---
Reprojection Threshold: 8.0px
[APE] SUCCESS: Error = 0.480 px | Markers: 3
[APE] SUCCESS: Error = 0.480 px | Markers: 3
[APE] SUCCESS: Error = 0.372 px | Markers: 3
[APE] SUCCESS: Error = 0.528 px | Markers: 3
[APE] SUCCESS: Error = 0.339 px | Markers: 3
[APE] SUCCESS: Error = 0.329 px | Markers: 3
[APE] SUCCESS: Error = 0.473 px | Markers: 3
[APE] SUCCESS: Error = 0.497 px | Markers: 3
[APE] SUCCESS: Error = 0.445 px | Markers: 3
[APE] SUCCESS: Error = 0.372 px | Markers: 3
[APE] SUCCESS: Error = 0.627 px | Markers: 4
[APE] SUCCESS: Error = 0.433 px | Markers: 3
[APE] SUCCESS: Error = 0.540 px | Markers: 3
[ICT] SUCCESS: Error = 0.603 px | Markers: 3
[APE] SUCCESS: Error = 1.022 px | Markers: 3
[APE] SUCCESS: Error = 0.618 px | Markers: 2
[ICT] SUCCESS: Error = 0.857 px | Markers: 2
[APE] SUCCESS: Error = 0.766 px | Markers: 2
[APE] SUCCESS: Error = 0.980 px | Markers: 4
[APE] SUCCESS: Error = 0.879 px | Markers: 2
[APE] SUCCESS: Error = 0.814 px

## APE and ICT with pentip drawing 

In [ ]:
import cv2
import numpy as np
import json
import pyrealsense2 as rs

# ================================
# 1. LOAD CAMERA & GEOMETRY
# ================================
calib = np.load(r"E:\Sanjay\Dodeca\realsense_color_intrinsics_1280x720.npz")
K = calib["camera_matrix"]
D = calib["dist_coeffs"]

# --- PEN TIP CALIBRATION ---Estimated pen-tip position in dodecahedron frame: [  11.2584853   -12.22415209 -176.66782922]   [17.05949194, 3.61900281, -167.5519252]
PEN_TIP_L = np.array([0.30466388 ,-1.39366957,184.99224307], dtype=np.float32)

with open(r"E:\Sanjay\Dodeca\Rigid\optimized_marker_object_poses_ITERATIVE.json", "r") as f:
    marker_geometry = json.load(f)

for key in marker_geometry:
    marker_geometry[key]["R"] = np.array(marker_geometry[key]["R"], dtype=np.float32)
    marker_geometry[key]["t"] = np.array(marker_geometry[key]["t"], dtype=np.float32)

# ==============================
# 2. SETTINGS & KALMAN INIT
# ==============================
REPROJECTION_THRESHOLD = 4.0  
LK_PARAMS = dict(winSize=(21, 21), maxLevel=4, 
                 criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))

drawing_points = [] # Stores tip path
MAX_DRAW_DISTANCE = 120 

kf = cv2.KalmanFilter(12, 6) # [x, y, z, rx, ry, rz, vx, vy, vz, vrx, vry, vrz]
kf.transitionMatrix = np.eye(12, dtype=np.float32)
for i in range(6):
    kf.transitionMatrix[i, i+6] = 1.0  # Constant velocity model

kf.measurementMatrix = np.zeros((6, 12), np.float32) # [x, y, z, rx, ry, rz]
for i in range(6):
    kf.measurementMatrix[i, i] = 1.0

kf.processNoiseCov = np.eye(12, dtype=np.float32) * 0.01 
kf.measurementNoiseCov = np.eye(6, dtype=np.float32) * 0.01
#kf.measurementNoiseCov[0:3, 0:3] *= 3.0   # tvec usually noisier
#kf.measurementNoiseCov[3:6, 3:6] *= 0.5   # rvec more stable
kf.errorCovPost = np.eye(12, dtype=np.float32)

# ==============================
# 3. CORE FUNCTIONS
# ==============================
detector_params = cv2.aruco.DetectorParameters()
#detector_params.cornerRefinementMethod = cv2.aruco.CORNER_REFINE_SUBPIX
#detector_params.cornerRefinementWinSize = 7
#detector_params.cornerRefinementMaxIterations = 30
#detector_params.cornerRefinementMinAccuracy = 0.1
aruco_detector = cv2.aruco.ArucoDetector(
    cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_6X6_250),
    detector_params
)

def build_correspondences(corners_dict, marker_geometry, marker_size=20.0):
    obj_pts, img_pts = [], []
    s = marker_size / 2.0
    lc = np.array([[-s,-s,0],[s,-s,0],[s,s,0],[-s,s,0]], dtype=np.float32)
    for m_id, img_corners in corners_dict.items():
        m_str = str(m_id)
        if m_str not in marker_geometry: continue
        R_m, t_m = marker_geometry[m_str]["R"], marker_geometry[m_str]["t"]
        for i in range(4):
            obj_pts.append(R_m @ lc[i] + t_m)
            img_pts.append(img_corners[i])
    return np.array(obj_pts, np.float32), np.array(img_pts, np.float32), len(obj_pts) >= 8

def run_ICT(prev_gray, curr_gray, prev_corners_dict):
    tracked_corners = {}
    if not prev_corners_dict: return tracked_corners
    marker_ids = list(prev_corners_dict.keys())
    prev_centers = np.array([np.mean(prev_corners_dict[mid], axis=0) for mid in marker_ids], dtype=np.float32).reshape(-1,1,2)
    next_centers, status, _ = cv2.calcOpticalFlowPyrLK(prev_gray, curr_gray, prev_centers, None, **LK_PARAMS)
    if next_centers is None or not any(status): return tracked_corners
    valid_idx = np.where(status.flatten() == 1)[0]
    if len(valid_idx) < 1: return tracked_corners
    vel_centers = (next_centers[valid_idx] - prev_centers[valid_idx]).reshape(-1, 2)
    vel_mags = np.linalg.norm(vel_centers, axis=1)
    mean_v, std_v = np.mean(vel_mags), np.std(vel_mags)
    for i in valid_idx:
        m_id = marker_ids[i]
        if abs(vel_mags[np.where(valid_idx==i)[0][0]] - mean_v) > 3 * std_v and len(valid_idx) > 2: continue
        p_corners = prev_corners_dict[m_id].reshape(-1, 1, 2).astype(np.float32)
        n_corners, c_status, _ = cv2.calcOpticalFlowPyrLK(prev_gray, curr_gray, p_corners, None, **LK_PARAMS)
        if n_corners is not None and all(c_status):
            c_vel = (n_corners - p_corners).reshape(-1, 2)
            c_vel_mag = np.linalg.norm(c_vel, axis=1)
            mean_c, std_c = np.mean(c_vel_mag), np.std(c_vel_mag)
            if np.all(np.abs(c_vel_mag - mean_c) <= (3 * std_c + 0.1)):
                tracked_corners[m_id] = n_corners.reshape(4, 2)
    return tracked_corners

def get_bbox(rvec, tvec, marker_geometry, K, D):
    all_pts = []
    for k in marker_geometry:
        s = 10.0 
        lc = np.array([[-s,-s,0],[s,-s,0],[s,s,0],[-s,s,0]], dtype=np.float32)
        for i in range(4): all_pts.append(marker_geometry[k]["R"] @ lc[i] + marker_geometry[k]["t"])
    proj, _ = cv2.projectPoints(np.array(all_pts), rvec, tvec, K, D)
    proj = proj.reshape(-1, 2)
    return int(np.min(proj[:,0])), int(np.min(proj[:,1])), int(np.max(proj[:,0])), int(np.max(proj[:,1]))

# ==============================
# 4. MAIN LOOP
# ==============================
pipeline = rs.pipeline()
config = rs.config()
config.enable_stream(rs.stream.color, 1280, 720, rs.format.bgr8, 30)
pipeline.start(config)

initialized, prev_gray, prev_corners = False, None, {}

try:
    while True:
        frames = pipeline.wait_for_frames()
        frame = np.asanyarray(frames.get_color_frame().get_data())
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        h, w = frame.shape[:2]

        prediction = kf.predict()
        pred_rvec, pred_tvec = prediction[:3], prediction[3:6]

        if initialized:
            try:
                xmin, ymin, xmax, ymax = get_bbox(pred_rvec, pred_tvec, marker_geometry, K, D)
                bw, bh = xmax - xmin, ymax - ymin
                xmin, xmax = max(0, xmin - bw), min(w, xmax + bw)
                ymin, ymax = max(0, ymin - bh), min(h, ymax + bh)
                roi_gray = gray[ymin:ymax, xmin:xmax]
            except: roi_gray, xmin, ymin = gray, 0, 0
        else: roi_gray, xmin, ymin = gray, 0, 0

        corners, ids, _ = aruco_detector.detectMarkers(roi_gray)
        current_corners_candidate, pose_found = {}, False
        ape_err, ict_err = 0.0, 0.0

        # --- APE SECTION ---
        if ids is not None:
            ids_list = ids.flatten().tolist()
            current_corners_candidate = {ids_list[i]: corners[i][0] + [xmin, ymin] for i in range(len(ids_list))}
            obj_p, img_p, ok = build_correspondences(current_corners_candidate, marker_geometry)
            if ok:
                succ, rvec, tvec = cv2.solvePnP(obj_p, img_p, K, D, rvec=pred_rvec.copy(), tvec=pred_tvec.copy(), 
                                                useExtrinsicGuess=initialized, flags=cv2.SOLVEPNP_ITERATIVE)
                if succ:
                    proj_p, _ = cv2.projectPoints(obj_p, rvec, tvec, K, D)
                    ape_err = np.mean(np.linalg.norm(img_p - proj_p.reshape(-1, 2), axis=1))
                    if ape_err < REPROJECTION_THRESHOLD:
                        pose_found, mode_text, current_err = True, "APE (ArUco)", ape_err
                        tip_distance = np.linalg.norm(PEN_TIP_L)
                        tip_err_est = current_err * (1 + tip_distance / 100.0)
                        print(f"[APE] SUCCESS | MarkerErr={ape_err:.3f}px | TipErr≈{tip_err_est:.3f}px | Markers={len(ids)}")
                    else: print(f"[APE] REJECTED: Error={ape_err:.3f} px")

        # --- ICT SECTION ---
        if not pose_found and initialized and prev_gray is not None:
            tracked_dict = run_ICT(prev_gray, gray, prev_corners)
            obj_p, img_p, ok = build_correspondences(tracked_dict, marker_geometry)
            if ok:
                succ, rvec, tvec = cv2.solvePnP(obj_p, img_p, K, D, rvec=pred_rvec.copy(), tvec=pred_tvec.copy(), 
                                                useExtrinsicGuess=True, flags=cv2.SOLVEPNP_ITERATIVE)
                if succ:
                    proj_p, _ = cv2.projectPoints(obj_p, rvec, tvec, K, D)
                    ict_err = np.mean(np.linalg.norm(img_p - proj_p.reshape(-1, 2), axis=1))
                    if ict_err < REPROJECTION_THRESHOLD:
                        current_corners_candidate, pose_found, mode_text, current_err = tracked_dict, True, "ICT (Optical Flow)", ict_err
                        tip_distance = np.linalg.norm(PEN_TIP_L)
                        tip_err_est = current_err * (1 + tip_distance / 100.0)
                        print(f"[ICT] SUCCESS | MarkerErr={ict_err:.3f}px | TipErr≈{tip_err_est:.3f}px | Markers={len(tracked_dict)}")
                    else: print(f"[ICT] REJECTED: Error={ict_err:.3f} px")

        # --- UPDATE & DRAWING ---
        if pose_found :
            measurement = np.concatenate((rvec, tvec)).astype(np.float32)
            if not initialized:
                kf.statePost[:6] = measurement
                initialized = True
            
            estimated = kf.correct(measurement)
            draw_r, draw_t = estimated[:3], estimated[3:6]
            prev_corners = current_corners_candidate
            
            #if stable_counter > STABLE_FRAMES: 
                # Project Pen Tip
            tip_2d, _ = cv2.projectPoints(PEN_TIP_L.reshape(1,3), draw_r, draw_t, K, D)
            tip_px = tuple(tip_2d[0].flatten().astype(int))

            # Trail Persistence Logic
            if not drawing_points or (drawing_points[-1] is not None and np.linalg.norm(np.array(tip_px) - np.array(drawing_points[-1])) < MAX_DRAW_DISTANCE):
                drawing_points.append(tip_px)

            # UI Panel Drawing
            cv2.rectangle(frame, (10, 10), (320, 130), (0, 0, 0), -1)
            cv2.putText(frame, f"MODE: {mode_text}", (20, 35), 0, 0.6, (0, 255, 0), 2)
            cv2.putText(frame, f"APE Err: {ape_err:.3f} px", (20, 60), 0, 0.5, (255, 255, 255), 1)
            cv2.putText(frame, f"ICT Err: {ict_err:.3f} px", (20, 85), 0, 0.5, (255, 255, 255), 1)
            cv2.putText(frame, f"Tip Err Est: {tip_err_est:.3f} px", (20, 110), 0, 0.5, (0, 255, 255), 1)
            
            cv2.drawFrameAxes(frame, K, D, draw_r, draw_t, 40)
            #if stable_counter > STABLE_FRAMES:
            #    cv2.circle(frame, tip_px, 5, (0,0,255), -1)
            cv2.circle(frame, tip_px, 5, (0, 0, 255), -1)
        else:
            if drawing_points and drawing_points[-1] is not None:
                drawing_points.append(None) # Break line if tracking lost
            if initialized: print("[LOG] Tracking Lost - Predicting via Kalman Only")

        # Render the trail
        for i in range(1, len(drawing_points)):
            if drawing_points[i-1] is None or drawing_points[i] is None: continue
            cv2.line(frame, drawing_points[i-1], drawing_points[i], (255, 0, 255), 2, cv2.LINE_AA)

        prev_gray = gray.copy()
        cv2.imshow("Dodecahedron Pen - Drawing & Logs", frame)
        
        key = cv2.waitKey(1)
        if key == 27: break
        elif key == ord('c'): drawing_points = [] # Clear trail

finally:
    pipeline.stop()
    cv2.destroyAllWindows()

[APE] SUCCESS | MarkerErr=0.486px | TipErr≈1.385px | Markers=4
[APE] SUCCESS | MarkerErr=0.584px | TipErr≈1.664px | Markers=4
[APE] SUCCESS | MarkerErr=0.646px | TipErr≈1.840px | Markers=4
[APE] SUCCESS | MarkerErr=0.719px | TipErr≈2.049px | Markers=4
[APE] SUCCESS | MarkerErr=0.399px | TipErr≈1.136px | Markers=3
[APE] SUCCESS | MarkerErr=0.451px | TipErr≈1.285px | Markers=4
[APE] SUCCESS | MarkerErr=0.368px | TipErr≈1.049px | Markers=3
[APE] SUCCESS | MarkerErr=0.659px | TipErr≈1.879px | Markers=4
[APE] SUCCESS | MarkerErr=0.612px | TipErr≈1.746px | Markers=4
[APE] SUCCESS | MarkerErr=0.674px | TipErr≈1.920px | Markers=4
[APE] SUCCESS | MarkerErr=0.769px | TipErr≈2.192px | Markers=4
[APE] SUCCESS | MarkerErr=0.539px | TipErr≈1.536px | Markers=4
[APE] SUCCESS | MarkerErr=0.590px | TipErr≈1.682px | Markers=4
[APE] SUCCESS | MarkerErr=1.055px | TipErr≈3.007px | Markers=4
[APE] SUCCESS | MarkerErr=0.834px | TipErr≈2.378px | Markers=4
[APE] SUCCESS | MarkerErr=0.771px | TipErr≈2.198px | Ma

In [3]:
import cv2
import numpy as np
import json
import pyrealsense2 as rs

# ================================
# 1. LOAD CAMERA & GEOMETRY
# ================================
calib = np.load(r"E:\Sanjay\Dodeca\realsense_color_intrinsics_1280x720.npz")
K = calib["camera_matrix"]
D = calib["dist_coeffs"]

# --- PEN TIP CALIBRATION ---Estimated pen-tip position in dodecahedron frame: [  11.2584853   -12.22415209 -176.66782922]   [17.05949194, 3.61900281, -167.5519252] [  0.30466388  -1.39366957 184.99224307]
PEN_TIP_L = np.array([0.30466388 ,-1.39366957,184.99224307], dtype=np.float32)

with open(r"E:\Sanjay\Dodeca\Rigid\optimized_marker_object_poses_ITERATIVE.json", "r") as f:
    marker_geometry = json.load(f)

for key in marker_geometry:
    marker_geometry[key]["R"] = np.array(marker_geometry[key]["R"], dtype=np.float32)
    marker_geometry[key]["t"] = np.array(marker_geometry[key]["t"], dtype=np.float32)

# ==============================
# 2. SETTINGS & KALMAN INIT
# ==============================
REPROJECTION_THRESHOLD = 4.0  
LK_PARAMS = dict(winSize=(21, 21), maxLevel=4, 
                 criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))

drawing_points = [] # Stores tip path
MAX_DRAW_DISTANCE = 120 

kf = cv2.KalmanFilter(12, 6) # [x, y, z, rx, ry, rz, vx, vy, vz, vrx, vry, vrz]
kf.transitionMatrix = np.eye(12, dtype=np.float32)
for i in range(6):
    kf.transitionMatrix[i, i+6] = 1.0  # Constant velocity model

kf.measurementMatrix = np.zeros((6, 12), np.float32) # [x, y, z, rx, ry, rz]
for i in range(6):
    kf.measurementMatrix[i, i] = 1.0

kf.processNoiseCov = np.eye(12, dtype=np.float32) * 0.01 
kf.measurementNoiseCov = np.eye(6, dtype=np.float32) * 0.1
#kf.measurementNoiseCov[0:3, 0:3] *= 3.0   # tvec usually noisier
#kf.measurementNoiseCov[3:6, 3:6] *= 0.5   # rvec more stable
kf.errorCovPost = np.eye(12, dtype=np.float32)

# ==============================
# 3. CORE FUNCTIONS
# ==============================
detector_params = cv2.aruco.DetectorParameters()
detector_params.cornerRefinementMethod = cv2.aruco.CORNER_REFINE_SUBPIX
detector_params.cornerRefinementWinSize = 7
detector_params.cornerRefinementMaxIterations = 30
detector_params.cornerRefinementMinAccuracy = 0.1
aruco_detector = cv2.aruco.ArucoDetector(
    cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_6X6_250),
    detector_params
)

def build_correspondences(corners_dict, marker_geometry, marker_size=20.0):
    obj_pts, img_pts = [], []
    s = marker_size / 2.0
    lc = np.array([[-s,-s,0],[s,-s,0],[s,s,0],[-s,s,0]], dtype=np.float32)
    for m_id, img_corners in corners_dict.items():
        m_str = str(m_id)
        if m_str not in marker_geometry: continue
        R_m, t_m = marker_geometry[m_str]["R"], marker_geometry[m_str]["t"]
        for i in range(4):
            obj_pts.append(R_m @ lc[i] + t_m)
            img_pts.append(img_corners[i])
    return np.array(obj_pts, np.float32), np.array(img_pts, np.float32), len(obj_pts) >= 8

def run_ICT(prev_gray, curr_gray, prev_corners_dict):
    tracked_corners = {}
    if not prev_corners_dict: return tracked_corners
    marker_ids = list(prev_corners_dict.keys())
    prev_centers = np.array([np.mean(prev_corners_dict[mid], axis=0) for mid in marker_ids], dtype=np.float32).reshape(-1,1,2)
    next_centers, status, _ = cv2.calcOpticalFlowPyrLK(prev_gray, curr_gray, prev_centers, None, **LK_PARAMS)
    if next_centers is None or not any(status): return tracked_corners
    valid_idx = np.where(status.flatten() == 1)[0]
    if len(valid_idx) < 1: return tracked_corners
    vel_centers = (next_centers[valid_idx] - prev_centers[valid_idx]).reshape(-1, 2)
    vel_mags = np.linalg.norm(vel_centers, axis=1)
    mean_v, std_v = np.mean(vel_mags), np.std(vel_mags)
    for i in valid_idx:
        m_id = marker_ids[i]
        if abs(vel_mags[np.where(valid_idx==i)[0][0]] - mean_v) > 3 * std_v and len(valid_idx) > 2: continue
        p_corners = prev_corners_dict[m_id].reshape(-1, 1, 2).astype(np.float32)
        n_corners, c_status, _ = cv2.calcOpticalFlowPyrLK(prev_gray, curr_gray, p_corners, None, **LK_PARAMS)
        if n_corners is not None and all(c_status):
            c_vel = (n_corners - p_corners).reshape(-1, 2)
            c_vel_mag = np.linalg.norm(c_vel, axis=1)
            mean_c, std_c = np.mean(c_vel_mag), np.std(c_vel_mag)
            if np.all(np.abs(c_vel_mag - mean_c) <= (3 * std_c + 0.1)):
                tracked_corners[m_id] = n_corners.reshape(4, 2)
    return tracked_corners

def get_bbox(rvec, tvec, marker_geometry, K, D):
    all_pts = []
    for k in marker_geometry:
        s = 10.0 
        lc = np.array([[-s,-s,0],[s,-s,0],[s,s,0],[-s,s,0]], dtype=np.float32)
        for i in range(4): all_pts.append(marker_geometry[k]["R"] @ lc[i] + marker_geometry[k]["t"])
    proj, _ = cv2.projectPoints(np.array(all_pts), rvec, tvec, K, D)
    proj = proj.reshape(-1, 2)
    return int(np.min(proj[:,0])), int(np.min(proj[:,1])), int(np.max(proj[:,0])), int(np.max(proj[:,1]))

# ==============================
# 4. MAIN LOOP
# ==============================
pipeline = rs.pipeline()
config = rs.config()
config.enable_stream(rs.stream.color, 1280, 720, rs.format.bgr8, 30)
pipeline.start(config)

initialized, prev_gray, prev_corners = False, None, {}

try:
    while True:
        frames = pipeline.wait_for_frames()
        frame = np.asanyarray(frames.get_color_frame().get_data())
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        h, w = frame.shape[:2]

        prediction = kf.predict()
        pred_rvec, pred_tvec = prediction[:3], prediction[3:6]

        if initialized:
            try:
                xmin, ymin, xmax, ymax = get_bbox(pred_rvec, pred_tvec, marker_geometry, K, D)
                bw, bh = xmax - xmin, ymax - ymin
                xmin, xmax = max(0, xmin - bw), min(w, xmax + bw)
                ymin, ymax = max(0, ymin - bh), min(h, ymax + bh)
                roi_gray = gray[ymin:ymax, xmin:xmax]
            except: roi_gray, xmin, ymin = gray, 0, 0
        else: roi_gray, xmin, ymin = gray, 0, 0

        corners, ids, _ = aruco_detector.detectMarkers(roi_gray)
        current_corners_candidate, pose_found = {}, False
        ape_err, ict_err = 0.0, 0.0
        ape_err_mm, ict_err_mm = 0.0, 0.0

        # --- APE SECTION ---
        if ids is not None:
            ids_list = ids.flatten().tolist()
            current_corners_candidate = {ids_list[i]: corners[i][0] + [xmin, ymin] for i in range(len(ids_list))}
            obj_p, img_p, ok = build_correspondences(current_corners_candidate, marker_geometry)
            if ok:
                succ, rvec, tvec, inliers = cv2.solvePnPRansac(obj_p, img_p, K, D, rvec=pred_rvec.copy(), tvec=pred_tvec.copy(), 
                                                flags=cv2.SOLVEPNP_ITERATIVE, reprojectionError=3.0, iterationsCount=50)
                if succ:
                    proj_p, _ = cv2.projectPoints(obj_p, rvec, tvec, K, D)
                    #ape_err = np.mean(np.linalg.norm(img_p - proj_p.reshape(-1, 2), axis=1))
                    proj_p = proj_p.reshape(-1,2)
                    if inliers is not None and len(inliers) > 0:
                        inlier_obj = obj_p[inliers.flatten()]
                        inlier_img = img_p[inliers.flatten()]
                        inlier_proj = proj_p[inliers.flatten()]
                        ape_err = np.mean(np.linalg.norm(inlier_img - inlier_proj, axis=1))
                    else:
                        ape_err = np.mean(np.linalg.norm(img_p - proj_p, axis=1))
                    R_ape = cv2.Rodrigues(rvec)[0]
                    camera_points_ape = (R_ape @ obj_p.T).T + tvec.ravel()
                    mean_z_ape = np.mean(np.abs(camera_points_ape[:,2]))
                    ape_err_mm = ape_err * mean_z_ape / K[0,0]
                    if ape_err < REPROJECTION_THRESHOLD:
                        pose_found, mode_text, current_err = True, "APE (ArUco)", ape_err
                        tip_distance = np.linalg.norm(PEN_TIP_L)
                        print(f"[APE] SUCCESS | MarkerErr={ape_err:.3f}px ({ape_err_mm:.3f}mm) | Markers={len(ids)}")
                    else: print(f"[APE] REJECTED: Error={ape_err:.3f} px")

        # --- ICT SECTION ---
        if not pose_found and initialized and prev_gray is not None:
            tracked_dict = run_ICT(prev_gray, gray, prev_corners)
            obj_p, img_p, ok = build_correspondences(tracked_dict, marker_geometry)
            if ok:
                succ, rvec, tvec, inliers = cv2.solvePnPRansac(obj_p, img_p, K, D, rvec=pred_rvec.copy(), tvec=pred_tvec.copy(), 
                                              flags=cv2.SOLVEPNP_ITERATIVE, reprojectionError=3.0, iterationsCount=50)
                if succ:
                    proj_p, _ = cv2.projectPoints(obj_p, rvec, tvec, K, D)
                    #ict_err = np.mean(np.linalg.norm(img_p - proj_p.reshape(-1, 2), axis=1))
                    proj_p = proj_p.reshape(-1,2)
                    if inliers is not None and len(inliers) > 0:
                        inlier_obj = obj_p[inliers.flatten()]
                        inlier_img = img_p[inliers.flatten()]
                        inlier_proj = proj_p[inliers.flatten()]
                        ict_err = np.mean(np.linalg.norm(inlier_img - inlier_proj, axis=1))
                    else:
                        ict_err = np.mean(np.linalg.norm(img_p - proj_p, axis=1))
                    R_ict = cv2.Rodrigues(rvec)[0]
                    camera_points_ict = (R_ict @ obj_p.T).T + tvec.ravel()
                    mean_z_ict = np.mean(np.abs(camera_points_ict[:,2]))
                    ict_err_mm = ict_err * mean_z_ict / K[0,0]
                    if ict_err < REPROJECTION_THRESHOLD:
                        current_corners_candidate, pose_found, mode_text, current_err = tracked_dict, True, "ICT (Optical Flow)", ict_err
                        tip_distance = np.linalg.norm(PEN_TIP_L)
                        print(f"[ICT] SUCCESS | MarkerErr={ict_err:.3f}px ({ict_err_mm:.3f}mm) | Markers={len(tracked_dict)}")
                    else: print(f"[ICT] REJECTED: Error={ict_err:.3f} px")

        # --- UPDATE & DRAWING ---
        if pose_found :
            measurement = np.concatenate((rvec, tvec)).astype(np.float32)
            prediction_state = prediction[:6]
            rot_diff_rad = 0.0
            trans_diff_mm = 0.0
            pen_tip_diff_mm = 0.0
            rot_err_mm = 0.0
            if initialized:
                rot_diff_rad = np.linalg.norm(measurement[:3] - prediction_state[:3])
                trans_diff_mm = np.linalg.norm(measurement[3:6] - prediction_state[3:6])
                tip_meas = np.matmul(cv2.Rodrigues(measurement[:3])[0], PEN_TIP_L) + measurement[3:6]
                tip_pred = np.matmul(cv2.Rodrigues(prediction_state[:3])[0], PEN_TIP_L) + prediction_state[3:6]
                pen_tip_diff_mm = np.linalg.norm(tip_meas - tip_pred)
                rot_err_mm = rot_diff_rad * tip_distance
            if not initialized:
                kf.statePost[:6] = measurement
                initialized = True
            
            estimated = kf.correct(measurement)
            draw_r, draw_t = estimated[:3], estimated[3:6]
            prev_corners = current_corners_candidate
            
            #if stable_counter > STABLE_FRAMES: 
                # Project Pen Tip
            tip_2d, _ = cv2.projectPoints(PEN_TIP_L.reshape(1,3), draw_r, draw_t, K, D)
            tip_px = tuple(tip_2d[0].flatten().astype(int))

            # Trail Persistence Logic
            if not drawing_points or (drawing_points[-1] is not None and np.linalg.norm(np.array(tip_px) - np.array(drawing_points[-1])) < MAX_DRAW_DISTANCE):
                drawing_points.append(tip_px)

            # UI Panel Drawing
            cv2.rectangle(frame, (10, 10), (320, 200), (0, 0, 0), -1)
            cv2.putText(frame, f"MODE: {mode_text}", (20, 35), 0, 0.6, (0, 255, 0), 2)
            cv2.putText(frame, f"APE Err: {ape_err:.3f}px /{ape_err_mm:.3f}mm", (20, 60), 0, 0.5, (255, 255, 255), 1)
            cv2.putText(frame, f"ICT Err: {ict_err:.3f}px /{ict_err_mm:.3f}mm", (20, 85), 0, 0.5, (255, 255, 255), 1)
            cv2.putText(frame, f"Trans Err: {trans_diff_mm:.3f} mm", (20, 110), 0, 0.5, (0, 255, 255), 1)
            cv2.putText(frame, f"Rot Err: {rot_err_mm:.3f} mm", (20, 135), 0, 0.5, (0, 255, 255), 1)
            cv2.putText(frame, f"Pen Tip Err: {pen_tip_diff_mm:.3f} mm", (20, 160), 0, 0.5, (0, 255, 255), 1)
            
            cv2.drawFrameAxes(frame, K, D, draw_r, draw_t, 40)
            #if stable_counter > STABLE_FRAMES:
            #    cv2.circle(frame, tip_px, 5, (0,0,255), -1)
            cv2.circle(frame, tip_px, 5, (0, 0, 255), -1)
        else:
            if drawing_points and drawing_points[-1] is not None:
                drawing_points.append(None) # Break line if tracking lost
            if initialized: print("[LOG] Tracking Lost - Predicting via Kalman Only")

        # Render the trail
        for i in range(1, len(drawing_points)):
            if drawing_points[i-1] is None or drawing_points[i] is None: continue
            cv2.line(frame, drawing_points[i-1], drawing_points[i], (255, 0, 255), 2, cv2.LINE_AA)

        prev_gray = gray.copy()
        cv2.imshow("Dodecahedron Pen - Drawing & Logs", frame)
        
        key = cv2.waitKey(1)
        if key == 27: break
        elif key == ord('c'): drawing_points = [] # Clear trail

finally:
    pipeline.stop()
    cv2.destroyAllWindows()

[APE] SUCCESS | MarkerErr=0.645px (0.384mm) | Markers=4
[APE] SUCCESS | MarkerErr=0.673px (0.397mm) | Markers=4
[APE] SUCCESS | MarkerErr=0.583px (0.345mm) | Markers=4
[APE] SUCCESS | MarkerErr=0.550px (0.324mm) | Markers=4
[APE] SUCCESS | MarkerErr=0.822px (0.484mm) | Markers=4
[APE] SUCCESS | MarkerErr=0.775px (0.456mm) | Markers=4
[APE] SUCCESS | MarkerErr=0.601px (0.354mm) | Markers=4
[APE] SUCCESS | MarkerErr=0.537px (0.316mm) | Markers=4
[APE] SUCCESS | MarkerErr=0.695px (0.409mm) | Markers=4
[APE] SUCCESS | MarkerErr=0.602px (0.354mm) | Markers=4
[APE] SUCCESS | MarkerErr=0.571px (0.337mm) | Markers=4
[APE] SUCCESS | MarkerErr=0.614px (0.361mm) | Markers=4
[APE] SUCCESS | MarkerErr=0.614px (0.361mm) | Markers=4
[APE] SUCCESS | MarkerErr=0.662px (0.388mm) | Markers=4
[APE] SUCCESS | MarkerErr=0.631px (0.369mm) | Markers=4
[APE] SUCCESS | MarkerErr=0.606px (0.356mm) | Markers=4
[APE] SUCCESS | MarkerErr=0.630px (0.369mm) | Markers=4
[APE] SUCCESS | MarkerErr=0.623px (0.366mm) | Ma